# Week 21: Multi-sensor fusion: GOES-East, GOES-West, Himawari-9

**Track:** Space GIS Architect (Expert)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/21/](https://launchdetect.com/academy/week/21/)  
**Track index:** [https://launchdetect.com/academy/space-gis-architect/](https://launchdetect.com/academy/space-gis-architect/)

---

_One satellite sees one hemisphere. Three satellites see almost the whole disk. Fusing them is non-trivial: different projections, different timestamps, different radiometric calibrations._


## Why this week matters

Multi-sensor fusion is what gives LaunchDetect global coverage. Three GEO satellites overlap each other; combining them right is the difference between covering the Pacific and dropping every Mahia launch.


## Learning objectives

By the end of this lab you will be able to:

- Reproject and align imagery from three different GEO satellites
- Handle the seam between GOES-West and Himawari-9 over the Pacific
- Build a hemispheric mosaic with consistent radiometric scaling
- Manage the time-sync challenge across asynchronous sensors


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio pystac-client boto3`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio pystac-client boto3


## The approach (and why)

Visualize the useful viewing extents of GOES-East, GOES-West, and Himawari-9 on one map. From there, the real work is reprojecting individual sensor frames into a common Pacific-centric mosaic.


In [ ]:
# Week 21: multi-sensor fusion — overlay GOES-East + GOES-West coverage extents.
import leafmap.foliumap as leafmap
from pyproj import Geod

geod = Geod(ellps="WGS84")

def viewable_polygon(sub_sat_lon: float, useful_arc_deg: float = 70.0) -> dict:
    """Approximate GEO satellite's useful viewable disk as a polygon on Earth."""
    coords = []
    for bearing in range(0, 361, 5):
        dest_lon, dest_lat, _ = geod.fwd(sub_sat_lon, 0, bearing, useful_arc_deg * 111_000)
        coords.append([dest_lon, dest_lat])
    return {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [coords]},
            "properties": {"sub_sat_lon": sub_sat_lon}}

m = leafmap.Map(center=[0, -50], zoom=2)
m.add_geojson(viewable_polygon(-75.2), style={"color": "#ff6b35", "fillOpacity": 0.15})  # GOES-19 East
m.add_geojson(viewable_polygon(-137.2), style={"color": "#4a9eff", "fillOpacity": 0.15}) # GOES-18 West
m.add_geojson(viewable_polygon(140.7), style={"color": "#5eda9e", "fillOpacity": 0.15})  # Himawari-9
m

# TODO: download time-matched GOES-18 + Himawari-9 Band 7 frames and stitch
# them into a hemispheric mosaic GeoTIFF.


## What just happened — and why it works

Each GEO satellite has a circular 'useful' footprint of about ±70° from its sub-satellite point. Beyond that, limb effects make the data marginal. Overlapping coverage at the seam between GOES-West and Himawari-9 lets you cross-calibrate the two sensors' brightness temperatures.


## Common gotchas

- Sensor radiometric calibration differs. GOES Band 7 and Himawari Band 7 (AHI Band 7) are nominally identical but in practice need cross-calibration coefficients.
- Time-sync at the seam is hard; the two satellites scan at slightly different wall-clock times.
- Antimeridian (180°) handling — use a Pacific-centric projection so the seam is continuous.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/21/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
